In [31]:
import numpy as np 
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
nltk.download('ktinker')
import re 
import string
import os
path= '../data/questions.csv'
df=pd.read_csv(path)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/avakash/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading ktinker: Package 'ktinker' not found in
[nltk_data]     index


In [ ]:
df=df.reset_index(drop=True)

In [33]:
device='cuda'

In [34]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [35]:
# df=df.drop(columns=['id','qid1','qid2'])
df=df.dropna()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            404348 non-null  int64 
 1   qid1          404348 non-null  int64 
 2   qid2          404348 non-null  int64 
 3   question1     404348 non-null  object
 4   question2     404348 non-null  object
 5   is_duplicate  404348 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 21.6+ MB


In [36]:
def icon1(l):
    p=str(l).replace(",",' ,') 
    p=p.replace("?"," ?")
    p=p.replace("/"," ") 
    p=p.replace("."," . ")
    p=p.replace('-',' - ')
    p=p.replace("("," ( ") 
    p=p.replace(")"," ) ") 
    p=p.replace("<"," < ") 
    p=p.replace(">"," > ") 
    p=p.replace("\""," \" ") 
    p=p.replace("\'"," \' ")
    p=re.sub(r'[^a-zA-Z\s]','',p)
    p=p.lower()
    return p.split()

In [37]:
df['transformation1']=df['question1'].apply(icon1)
df['transformation2']=df['question2'].apply(icon1)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               404348 non-null  int64 
 1   qid1             404348 non-null  int64 
 2   qid2             404348 non-null  int64 
 3   question1        404348 non-null  object
 4   question2        404348 non-null  object
 5   is_duplicate     404348 non-null  int64 
 6   transformation1  404348 non-null  object
 7   transformation2  404348 non-null  object
dtypes: int64(4), object(4)
memory usage: 27.8+ MB


In [39]:
vocab=set()
for i in df.question1:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
for i in df.question2:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
vocab=list(vocab)

In [40]:
len(vocab)

80808

In [41]:
vocab[:10]

['deserting',
 'mastered',
 'vultimate',
 'hearse',
 'secularising',
 'preventive',
 'hungrier',
 'provable',
 'katchatheevu',
 'fastrak']

In [42]:
id2voc={}
voc2id={}
idx=0
for i in vocab:
    id2voc[idx]=i
    voc2id[i]=idx
    idx+=1
id2voc[idx]='<UNK>'
voc2id['<UNK>']=idx


In [43]:
len(voc2id)

80809

In [44]:
import torch
import torch.nn as nn
vocab_size=len(voc2id)
embedding_dim=128

In [45]:
E=nn.Embedding(vocab_size,embedding_dim).to(device)

In [46]:
sample=df.question1[0]
sample=icon1(sample)
sample_test=[voc2id[i] for i in sample]
sample_test

[63111,
 33193,
 25493,
 55305,
 80200,
 55305,
 43110,
 23156,
 50310,
 54765,
 68688,
 39431,
 54765,
 34703]

In [47]:
x=torch.tensor(sample_test)
embed=E(x.to(device))
print(embed.shape)

torch.Size([14, 128])


In [48]:
avg=embed.mean(dim=0)
avg.shape

torch.Size([128])

In [49]:
def floss(i1,i2):
    return nn.functional.cosine_similarity(i1,i2)

class Embeds(nn.Module):
    def __init__(self,voc2id,E):
        super().__init__()
        self.voc2id=voc2id
        self.E=E
    def embedding(self,x):
        x=self.icon1(x)
        x=[self.voc2id.get(i,self.voc2id['<UNK>']) for i in x]
        return self.E(torch.tensor(x, dtype=torch.long).to(device))
    def icon1(self,l):
        p=str(l).replace(",",' ,') 
        p=p.replace("?"," ?")
        p=p.replace("/"," ") 
        p=p.replace("."," . ")
        p=p.replace('-',' - ')
        p=p.replace("("," ( ") 
        p=p.replace(")"," ) ") 
        p=p.replace("<"," < ") 
        p=p.replace(">"," > ") 
        p=p.replace("\""," \" ") 
        p=re.sub(r'[^a-zA-Z\s]','',p)
        p=p.replace("\'"," \' ")
        p=p.lower() 
        p=p.split() 
        return p
    def forward(self,x):
        emb=self.embedding(x)
        pooled=emb.mean(dim=0)
        return pooled
    

In [50]:
model=Embeds(voc2id=voc2id,E=E).to(device)


In [51]:
model.eval()
x="What we could do to explore stock"
y="what is steps to invest in stocks"
op=model(x)
op1=model(y)
op,op.shape

(tensor([-3.3735e-01,  8.7835e-01,  5.3164e-02,  7.3412e-01, -7.6776e-02,
         -5.1380e-01,  3.6153e-01,  8.6026e-01, -4.1670e-01, -1.3721e-01,
         -9.8979e-02,  8.0209e-01,  3.1811e-01, -7.3937e-01, -1.4666e-01,
         -1.3725e-01, -3.2514e-01,  1.2991e-01,  2.6639e-01, -8.3565e-01,
          3.6006e-02,  1.1152e+00,  1.7430e-01, -3.9429e-01,  4.7257e-01,
          5.1952e-02, -7.3033e-02, -6.6201e-01,  3.6234e-02, -4.0631e-01,
         -2.5438e-01, -4.4078e-01,  8.2725e-02,  2.8014e-01, -7.4835e-03,
          1.1311e-02,  1.3276e-01,  8.5436e-03,  1.8467e-01, -1.3804e-01,
         -6.6404e-02, -3.0439e-01,  3.7190e-01, -6.0671e-01, -3.4829e-01,
          9.2249e-01, -6.5687e-02, -5.3807e-01,  8.6784e-02,  2.5665e-01,
         -4.2222e-01,  1.2208e-01, -6.7763e-02,  2.2786e-01,  5.7927e-01,
          1.7727e-01,  7.0549e-01, -4.2965e-01,  4.0143e-01, -8.4838e-02,
          2.8289e-01, -6.6525e-01,  1.8999e-01, -1.1193e-01, -5.2968e-01,
          1.5281e-01, -3.0634e-01, -9.

In [52]:
similarit=nn.functional.cosine_similarity(
    op1.unsqueeze(0),
    op.unsqueeze(0)
)
similarit

tensor([0.2652], device='cuda:0', grad_fn=<SumBackward1>)

In [53]:
df.loc[df['is_duplicate']==0,'is_duplicate']=-1

In [54]:
import torch.optim as optim
lossfn=nn.CosineEmbeddingLoss()
opt=optim.Adam(model.parameters(),lr=0.001)

In [59]:
def valid_sentence(sentence):
    return len(sentence) > 0

In [ ]:
pairs=[]
removed=0
for i in range(len(df)):
    q1 = df.transformation1.iloc[i]
    q2 = df.transformation2.iloc[i]
    label = df.is_duplicate.iloc[i]
    if valid_sentence(q1) and valid_sentence(q2):
        pairs.append((q1, q2, label))
    else:
        print(removed)
        removed += 1
print("Removed:", removed)
print("Remaining:", len(pairs))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
Removed: 28
Remaining: 404320


In [145]:
pairs[0]

('What is the step by step guide to invest in share market in india?',
 'What is the step by step guide to invest in share market?',
 np.int64(-1))

In [63]:
epoch=1
model.train()
for e in range(epoch):
    total_loss=0
    for s1,s2,label in pairs:
        e1=model(s1)
        e2=model(s2)
        target=torch.tensor(label).float().to(device)
        loss=lossfn(e1.unsqueeze(0),e2.unsqueeze(0),target.unsqueeze(0))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss+=loss.item()
    print(f'epoch {e} : {total_loss}')

KeyboardInterrupt: 

In [178]:
e1.device

device(type='cuda', index=0)

In [64]:
import torch
import pickle
torch.save(model.state_dict(), "embed_model.pth")
with open("voc2id.pkl", "wb") as f:
    pickle.dump(voc2id,f)
with open("id2voc.pkl","wb") as f:
    pickle.dump(id2voc,f)